# CNN Training — CESM2-LE JJA SST → September Slowdown

Trains the binary CNN over `N_SPLITS` TVT splits × `N_SEEDS` random seeds and produces:
0. Learning curves
1. Precision–recall curve (with threshold selection)
2. Confusion matrices (train + test)
3. Evaluation metrics across all seeds/splits
4. Slowdown time series per ensemble member

**Set `N_SPLITS` and `N_SEEDS` in the Config cell, then Run All.**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths
from src.cnn.splits import load_tvt_split
from src.cnn.model import build_cnn, compute_metrics, METRIC_NAMES
from src.cnn.train import (
    set_seed, compute_class_weights, train_model,
    predict_splits, collect_metrics_dataset,
    save_model, save_metrics_dataset,
)

## Config
Edit these values before running.

In [ ]:
# ── How many splits / seeds to run ───────────────────────────────────────────
N_SPLITS   = 3          # max 9
N_SEEDS    = 3          # random seeds per split
BASE_SEED  = 42         # seeds: BASE_SEED, BASE_SEED+1, …

# ── Architecture ─────────────────────────────────────────────────────────────
RL2  = 1e-5
DROP = 0.2

# ── Class weights ────────────────────────────────────────────────────────────
FRACT_WEIGHT = 1.5      # upward multiplier on the minority (slowdown) class

# ── Training hyperparams ─────────────────────────────────────────────────────
TRAIN_CONFIG = {
    'learning_rate': 1e-4,
    'num_epochs':    50,
    'batch_size':    120,
    'patience':      10,
    'focal_alpha':   0.75,
    'focal_gamma':   2.0,
}

# ── Dataset ───────────────────────────────────────────────────────────────────
BLOCK_SIZE  = 10        # ensemble members per TVT block
START_YEAR  = 1990      # first year in the loaded splits

METRICS_DIR = paths.RESULTS_DIR / 'metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## Training loop
Trains all splits × seeds. Results are stored in `results` for downstream plotting.

In [ ]:
import json

# results[split_idx] = {
#   'histories':     list[History]  (one per seed)
#   'y_scores_runs': list[dict]     (one dict per seed, keys: train/val/test)
#   'y_true':        dict           (keys: train/val/test)
#   'ds_metrics':    xr.Dataset
# }
results = {}

paths.LOGS_DIR.mkdir(parents=True, exist_ok=True)

for split_idx in range(N_SPLITS):
    print(f'\n{"─"*60}')
    print(f'Split {split_idx}')
    print(f'{"─"*60}')

    split_path = paths.tvt_split_path(split_idx)
    if not split_path.exists():
        raise FileNotFoundError(
            f'TVT split file not found:\n  {split_path}\n'
            f'Run scripts/03_cesm2le_tvt_splits.py first.'
        )
    split = load_tvt_split(split_path)

    x_tr = split['sst_tr'][:, :, :, np.newaxis]
    x_va = split['sst_va'][:, :, :, np.newaxis]
    x_te = split['sst_te'][:, :, :, np.newaxis]
    y_tr = split['slow_tr']
    y_va = split['slow_va']
    y_te = split['slow_te']

    nx, ny, nch = x_tr.shape[1], x_tr.shape[2], x_tr.shape[3]
    y_true = {'train': y_tr, 'val': y_va, 'test': y_te}

    print(f'  Train: {x_tr.shape}  prevalence={y_tr.mean():.3f}')
    print(f'  Val  : {x_va.shape}  prevalence={y_va.mean():.3f}')
    print(f'  Test : {x_te.shape}  prevalence={y_te.mean():.3f}')

    histories      = []
    y_scores_runs  = []

    for run_idx in range(N_SEEDS):
        seed = BASE_SEED + run_idx
        print(f'  seed={seed} ...', end=' ', flush=True)

        set_seed(seed)
        cw    = compute_class_weights(y_tr, fract_weight=FRACT_WEIGHT)
        model = build_cnn(nx, ny, nch, rl2=RL2, drop=DROP)
        model, history = train_model(
            model, x_tr, y_tr, x_va, y_va,
            config=TRAIN_CONFIG, class_weights=cw,
        )

        n_ep    = len(history.history['loss'])
        val_loss = history.history['val_loss'][-1]
        print(f'stopped at epoch {n_ep}, val_loss={val_loss:.4f}')

        save_model(model, paths.MODELS_DIR, split_idx, run_idx)

        # Save training history as JSON for use in cnn_predict.ipynb
        hist_path = paths.LOGS_DIR / f'history_split{split_idx}_run{run_idx}.json'
        with open(hist_path, 'w') as f:
            json.dump({k: [float(v) for v in vals]
                       for k, vals in history.history.items()}, f)

        histories.append(history)
        y_scores_runs.append(predict_splits(model, x_tr, x_va, x_te))

    ds_metrics = collect_metrics_dataset(y_true, y_scores_runs)
    save_metrics_dataset(ds_metrics, METRICS_DIR, split_idx)

    results[split_idx] = {
        'histories':     histories,
        'y_scores_runs': y_scores_runs,
        'y_true':        y_true,
        'ds_metrics':    ds_metrics,
    }

print('\nTraining complete.')

---
## Visualization

**Plots 0–2 and 4** show results for a single split × seed — set `PLOT_SPLIT` and `PLOT_SEED` below.  
**Plot 3** aggregates across all trained splits and seeds.

In [ ]:
PLOT_SPLIT = 0    # which split to visualize (0 to N_SPLITS-1)
PLOT_SEED  = 0    # which seed to visualize  (0 to N_SEEDS-1)

# Convenience references
res      = results[PLOT_SPLIT]
history  = res['histories'][PLOT_SEED]
y_scores = res['y_scores_runs'][PLOT_SEED]
y_true   = res['y_true']

### 0 — Learning curves

In [ ]:
hist = history.history
epochs = np.arange(1, len(hist['loss']) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, hist['loss'],     color='steelblue',  label='train loss')
ax.plot(epochs, hist['val_loss'], color='steelblue',  label='val loss',
        linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Learning curve  (split {PLOT_SPLIT}, seed {PLOT_SEED})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1 — Precision–Recall curve & threshold selection

In [ ]:
from sklearn.metrics import f1_score as sk_f1

y_s = y_scores['test']
y_t = y_true['test']

precision, recall, thresholds = precision_recall_curve(y_t, y_s)

# F1 at each threshold
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)

# Threshold at precision–recall intersection (same rule as compute_metrics)
best_idx = np.argmin(np.abs(precision[:-1] - recall[:-1]))
best_thr = thresholds[best_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precision[:-1], label='Precision', color='steelblue')
ax.plot(thresholds, recall[:-1],    label='Recall',    color='darkorange')
ax.plot(thresholds, f1_scores,      label='F1',        color='seagreen')
ax.axvline(best_thr, color='black', linewidth=2, label=f'threshold = {best_thr:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Precision–Recall–F1 vs threshold  (split {PLOT_SPLIT}, seed {PLOT_SEED}, test)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Selected threshold: {best_thr:.4f}  '
      f'(Precision={precision[best_idx]:.3f}, Recall={recall[best_idx]:.3f})')

### 2 — Confusion matrices (train + test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, split_name in zip(axes, ['train', 'test']):
    yt = y_true[split_name]
    ys = y_scores[split_name]

    # Recompute threshold using this split's PR curve
    prec, rec, thr = precision_recall_curve(yt, ys)
    if thr.size > 0:
        thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))]
    else:
        thr_best = 0.5

    y_pred = (ys >= thr_best).astype(int)
    cm     = confusion_matrix(yt, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Negative', 'Positive']
    )
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(f'{split_name.capitalize()}  (split {PLOT_SPLIT}, seed {PLOT_SEED})')

plt.tight_layout()
plt.show()

### 3 — Evaluation metrics across all seeds & splits
Mean value per metric on the test split, with individual seed/split values overlaid.

In [ ]:
# Collect metric_value arrays across all trained splits
# Shape per split: (n_metrics, n_splits_dim, n_seeds) — we want the 'test' split dim
all_vals = []   # list of (n_metrics, n_seeds) arrays
for sidx in range(N_SPLITS):
    ds = results[sidx]['ds_metrics']
    all_vals.append(ds['metric_value'].sel(split='test').values)  # (n_metrics, n_seeds)

# Stack: (n_splits, n_metrics, n_seeds) → flatten last two → (n_metrics, n_splits*n_seeds)
all_vals = np.stack(all_vals, axis=0)              # (n_splits, n_metrics, n_seeds)
all_vals = all_vals.transpose(1, 0, 2)             # (n_metrics, n_splits, n_seeds)
n_metrics = all_vals.shape[0]

# Metrics to display (skip internal-only ones)
DISPLAY_METRICS = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUPRC', 'AUROC', 'MCC']
display_idx     = [METRIC_NAMES.index(m) for m in DISPLAY_METRICS]

vals_display = all_vals[display_idx, :, :]         # (n_display, n_splits, n_seeds)
vals_flat    = vals_display.reshape(len(DISPLAY_METRICS), -1)  # (n_display, n_splits*n_seeds)
means        = np.nanmean(vals_flat, axis=1)

fig, ax = plt.subplots(figsize=(9, 5))

for i, (m, mean_val) in enumerate(zip(DISPLAY_METRICS, means)):
    jitter = np.random.uniform(-0.15, 0.15, size=vals_flat.shape[1])
    ax.scatter(
        np.full(vals_flat.shape[1], i) + jitter,
        vals_flat[i],
        color='steelblue', alpha=0.5, s=25, zorder=2
    )
    ax.plot([i - 0.3, i + 0.3], [mean_val, mean_val],
            color='darkred', linewidth=2.5, zorder=3)

ax.set_xticks(range(len(DISPLAY_METRICS)))
ax.set_xticklabels(DISPLAY_METRICS)
ax.set_ylabel('Score (%)')
ax.set_title(f'TESTING  —  metrics across {N_SPLITS} splits × {N_SEEDS} seeds')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4 — Slowdown time series per ensemble member
Shows actual slowdowns (gray), correct predictions (green), and incorrect predictions (red) for each test-block member.

In [ ]:
# ── Recover per-member structure from flattened test arrays ──────────────────
yt = y_true['test']                        # (n_test_samples,)
ys = y_scores['test']

# Threshold (same rule as compute_metrics / PR curve)
prec, rec, thr = precision_recall_curve(yt, ys)
thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
y_pred   = (ys >= thr_best).astype(int)

# Reshape: (BLOCK_SIZE, nyears)
nyears     = yt.shape[0] // BLOCK_SIZE
years      = np.arange(START_YEAR, START_YEAR + nyears)
yt_2d      = yt.reshape(BLOCK_SIZE, nyears)
ypred_2d   = y_pred.reshape(BLOCK_SIZE, nyears)

# Global member indices (test block = split_idx * BLOCK_SIZE ... + BLOCK_SIZE)
test_block_start = PLOT_SPLIT * BLOCK_SIZE
member_labels    = np.arange(test_block_start, test_block_start + BLOCK_SIZE)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))

for m_idx, m_label in enumerate(member_labels):
    actual   = yt_2d[m_idx]      # (nyears,)
    predicted = ypred_2d[m_idx]

    # Horizontal guide line
    ax.axhline(m_label, color='lightgray', linewidth=0.6, zorder=0)

    # Actual slowdowns — gray filled
    actual_yrs = years[actual == 1]
    ax.scatter(actual_yrs, np.full_like(actual_yrs, m_label, dtype=float),
               color='dimgray', s=70, zorder=2,
               label='actual slowdowns' if m_idx == 0 else '')

    # Correct predictions (TP: pred=1, true=1)
    tp_yrs = years[(predicted == 1) & (actual == 1)]
    ax.scatter(tp_yrs, np.full_like(tp_yrs, m_label, dtype=float),
               color='mediumseagreen', s=30, zorder=3,
               label='correct predictions' if m_idx == 0 else '')

    # Incorrect predictions (FP: pred=1, true=0 OR FN: pred=0, true=1)
    err_yrs = years[(predicted == 1) & (actual == 0)]
    ax.scatter(err_yrs, np.full_like(err_yrs, m_label, dtype=float),
               color='lightcoral', s=30, zorder=3,
               label='incorrect predictions' if m_idx == 0 else '')

ax.set_xlabel('Year')
ax.set_ylabel('Ensemble member no.')
ax.set_xlim(years[0] - 1, years[-1] + 1)
ax.set_yticks(member_labels)
ax.set_title(f'Slowdown predictions — test block (split {PLOT_SPLIT}, seed {PLOT_SEED})')
ax.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.12),
          frameon=False, markerscale=1.4)
ax.grid(False)
plt.tight_layout()
plt.show()